In [64]:
# Define source path for journal results
SOURCE_PATH = "../../save_and_results/old/journal/"

In [65]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd
from utils.display_tools import load_best_forecasts

In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
lot = pickle.load(open("../../save_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../../save_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../../save_and_results/cache_exogs.p", "rb"))

chronos = pd.read_csv(SOURCE_PATH + "chronos/Moehne_desc1_chronos_zero_shot_chronos.csv", index_col=0)
tfm = pd.read_csv(SOURCE_PATH + "/tfm/Moehne_desc1_tfm_zero_shot_chronos.csv", index_col=0)

d = data["Moehne"]["desc1"]
y_ = d.loc[d.index.year == 2020]

### Table for the PS points

In [69]:
forec = {}
model_stack = {}
for x in [str(n) for n in range(0,4)]:
    model_spec, forecasts = load_best_forecasts(SOURCE_PATH  + x, direction="desc1")
    model_stack[x] = model_spec

    if (x == "1")  or (x == "2"):
        counter = [x for x in forecasts.keys() if "sarimax" not in x]
        counter = [x for x in counter if "var" not in x]
    else:
        counter = forecasts.keys()
    forec[x] = {x: np.abs(forecasts[x] - y_.values).mean() for x in counter}
    forec[x] = pd.concat(forec[x], axis=1).round(3)



chronos = pd.read_csv(SOURCE_PATH + "chronos/Moehne_desc1_chronos_zero_shot_chronos.csv", index_col=0)
tfm = pd.read_csv(SOURCE_PATH + "tfm/Moehne_desc1_tfm_zero_shot_chronos.csv", index_col=0)
# append foundational models
forec["3"]["chronos"] = np.abs(y_ - chronos.values[:-1]).mean().round(3)
forec["3"]["tfm"] = np.abs(y_ - tfm.values[:-1]).mean().round(3)

forec["0"]["chronos"] = np.abs(y_ - chronos.values[:-1]).mean().round(3)
forec["0"]["tfm"] = np.abs(y_ - tfm.values[:-1]).mean().round(3)

In [70]:
final_naming = ["Full search", "Baseline", "Full Exogenous","Univariate"]
final = []
for n,x in enumerate(forec.keys()):
    print(x)
    a = forec[x].min(axis=1).round(2)
    naming = [forec[x].T.sort_values(y).index[0] for y in a.index]
    naming = [x[:3] if x != "sarimax" else "ari" for x in naming ]
    a = "\makecell{" + a.astype(str) + " // (" + naming +  ")}"

    final.append(a)
final = pd.concat(final, axis=1)
final.columns = final_naming

0
1
2
3


<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_93736/3587143499.py:8: SyntaxWarning: invalid escape sequence '\m'
  a = "\makecell{" + a.astype(str) + " // (" + naming +  ")}"


In [71]:
for x in final.columns:
    final[x] = final[x].str.replace("sar", "ari")

In [72]:
final = final[["Baseline","Full Exogenous", "Univariate", "Full search"]]
final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_93736/1023676759.py:2: SyntaxWarning: invalid escape sequence '\m'
  final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]
/tmp/ipykernel_93736/1023676759.py:2: SyntaxWarning: invalid escape sequence '\m'
  final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]


In [73]:
final.T

,10905,10961,10982,11038,11161,11176,11208,11243
Baseline,\makecell{1.52 // (lin)},\makecell{1.02 // (lin)},\makecell{1.18 // (lin)},\makecell{3.85 // (lin)},\makecell{2.05 // (lin)},\makecell{1.21 // (lin)},\makecell{1.56 // (lin)},\makecell{3.16 // (lin)}
\makecell{Baseline + \ Exogenous.},\makecell{1.56 // (lin)},\makecell{0.98 // (lin)},\makecell{1.12 // (lin)},\makecell{3.64 // (lin)},\makecell{2.09 // (lin)},\makecell{1.0 // (lin)},\makecell{1.42 // (lin)},\makecell{3.26 // (lin)}
Univariate,\makecell{1.34 // (ari)},\makecell{0.81 // (lin)},\makecell{0.97 // (ada)},\makecell{3.68 // (ada)},\makecell{2.05 // (ari)},\makecell{1.01 // (lin)},\makecell{1.39 // (ari)},\makecell{3.11 // (ada)}
\makecell{Full model \ search space},\makecell{1.5 // (tfm)},\makecell{0.87 // (chr)},\makecell{1.08 // (lin)},\makecell{3.59 // (lin)},\makecell{2.04 // (for)},\makecell{1.06 // (var)},\makecell{1.42 // (lin)},\makecell{3.21 // (ada)}


In [74]:
print(final.T.to_latex())

\begin{tabular}{lllllllll}
\toprule
 & 10905 & 10961 & 10982 & 11038 & 11161 & 11176 & 11208 & 11243 \\
\midrule
Baseline & \makecell{1.52 // (lin)} & \makecell{1.02 // (lin)} & \makecell{1.18 // (lin)} & \makecell{3.85 // (lin)} & \makecell{2.05 // (lin)} & \makecell{1.21 // (lin)} & \makecell{1.56 // (lin)} & \makecell{3.16 // (lin)} \\
\makecell{Baseline + \ Exogenous.} & \makecell{1.56 // (lin)} & \makecell{0.98 // (lin)} & \makecell{1.12 // (lin)} & \makecell{3.64 // (lin)} & \makecell{2.09 // (lin)} & \makecell{1.0 // (lin)} & \makecell{1.42 // (lin)} & \makecell{3.26 // (lin)} \\
Univariate & \makecell{1.34 // (ari)} & \makecell{0.81 // (lin)} & \makecell{0.97 // (ada)} & \makecell{3.68 // (ada)} & \makecell{2.05 // (ari)} & \makecell{1.01 // (lin)} & \makecell{1.39 // (ari)} & \makecell{3.11 // (ada)} \\
\makecell{Full model \ search space} & \makecell{1.5 // (tfm)} & \makecell{0.87 // (chr)} & \makecell{1.08 // (lin)} & \makecell{3.59 // (lin)} & \makecell{2.04 // (for)} & \ma